In [1]:
import pyro
import pyro.distributions as dist

from pyro.nn import PyroSample
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO
from pgmpy.parameter.BayesianFunctionalRegression import BayesianFunctionalRegression
from pgmpy.parameter._base import BaseParameter
from skpro.distributions.normal import Normal as SkproNormal
from torch import nn
from pyro.nn import PyroModule
from pyro.nn import PyroSample

from pyro.infer import Predictive
import os
from functools import partial
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pyro.set_rng_seed(1)

%matplotlib inline
plt.style.use('default')


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_URL = "https://github.com/pyro-ppl/datasets/blob/master/rugged_data.csv?raw=true"
data = pd.read_csv(DATA_URL, encoding="ISO-8859-1")
df = data[["cont_africa", "rugged", "rgdppc_2000"]]
df = df[np.isfinite(df.rgdppc_2000)]
df["rgdppc_2000"] = np.log(df["rgdppc_2000"])

# Dataset: Add a feature to capture the interaction between "cont_africa" and "rugged"
df["cont_africa_x_rugged"] = df["cont_africa"] * df["rugged"]
data = torch.tensor(df[["cont_africa", "rugged", "cont_africa_x_rugged", "rgdppc_2000"]].values,
                        dtype=torch.float)

X = df[["cont_africa","rugged","cont_africa_x_rugged"]]
y = df[["rgdppc_2000"]]

x_data, y_data = data[:, :-1], data[:, -1]


In [3]:
class UserCustom(PyroModule):
    def __init__(self, in_features, mu=0.0, sigma=1.0):
        super().__init__()

        self.linear = PyroModule[nn.Linear](in_features, 1)

        self.linear.weight = PyroSample(
            dist.Normal(mu, sigma)
            .expand([1, in_features])
            .to_event(2)
        )

        self.linear.bias = PyroSample(
            dist.Normal(mu, sigma)
            .expand([1])
            .to_event(1)
        )

    def forward(self, x, y=None):
        obs_scale = pyro.sample(
            "obs_scale",
            dist.HalfNormal(2.0)
        )

        # Student-t
        df = 2.0 + pyro.sample(
            "df_minus_two",
            dist.Exponential(1.0)
        )

        mean = self.linear(x).squeeze(-1)

        with pyro.plate("data", x.shape[0]):
            pyro.sample(
                "obs",
                dist.StudentT(
                    df=df,
                    loc=mean,
                    scale=obs_scale,
                ),
                obs=y,
            )

        return mean


In [4]:
def pyro2skpro():
    def converter(y_samples, num_samples, X_array, index, columns):
        n_instances = X_array.shape[0]

        y_samples = y_samples.reshape(num_samples, n_instances)

        pred_mean = y_samples.mean(dim=0)
        pred_sigma = y_samples.std(dim=0, unbiased=False)

        eps = torch.finfo(pred_sigma.dtype).eps
        pred_sigma = pred_sigma.clamp_min(eps)

        pred_mean = pred_mean.detach().cpu().numpy()
        pred_sigma = pred_sigma.detach().cpu().numpy()

        return SkproNormal(
            mu=pred_mean.reshape(-1, 1),
            sigma=pred_sigma.reshape(-1, 1),
            index=index,
            columns=pd.Index(["y"]),
        )

    return converter


In [5]:
model = UserCustom(in_features=3)
guide = AutoDiagonalNormal(model)


In [6]:
linear_model = BayesianFunctionalRegression(
    model=model,
    guide=guide,
    converter=pyro2skpro(),
    num_iterations=500, 
    lr=0.03,
    posterior_samples=100,
) 


In [7]:
linear_model.fit(X, y)


[iteration 0001] loss: 9.0017
[iteration 0101] loss: 3.1727
[iteration 0201] loss: 2.8238
[iteration 0301] loss: 2.4086
[iteration 0401] loss: 2.0518


BayesianFunctionalRegression(converter=<function pyro2skpro.<locals>.converter at 0x0000017B644C04A0>,
                             guide=AutoDiagonalNormal(),
                             model=UserCustom(
  (linear): PyroLinear(in_features=3, out_features=1, bias=True)
),
                             num_iterations=500, posterior_samples=100)

In [8]:
median = linear_model.guide.median(x_data, y_data)

for name, value in median.items():
    print(name, ": ", value.detach().cpu().numpy())

# obs_scale :  0.9491169
# df_minus_two :  2.3301017
# linear.weight :  [[-0.93793654  0.19244921 -0.1710268 ]]
# linear.bias :  [8.4146805]

# regression function: y∼Normal(8.998347−1.7521899x1−0.12638536x2+0.22601697x1x2,1.0184335)


obs_scale :  0.9572112
df_minus_two :  1.8282388
linear.weight :  [[-1.0648167   0.2050247  -0.11136866]]
linear.bias :  [8.474473]


In [9]:
import pandas as pd

X_df = pd.DataFrame(
    x_data[:5].detach().cpu().numpy(),
    columns=["x1", "x2", "x3"],
)

pred_dist = linear_model.predict_proba(X_df)


In [10]:
pred_dist


Normal(columns=Index(['y'], dtype='object'),
       index=RangeIndex(start=0, stop=5, step=1),
       mu=array([[7.504105],
       [9.239355],
       [8.546478],
       [8.49471 ],
       [9.441691]], dtype=float32),
       sigma=array([[1.5321285],
       [1.1872301],
       [1.3295486],
       [1.3096495],
       [3.5523667]], dtype=float32))

In [24]:
pred_dist.mu.shape


(5, 1)